# Matz et al., Nature Immunology, 2026 (transcriptomics)

Author: Julian Q. Zhou

Docker container: `julianqz/wu_cimm:ref_0.2.2_lsf` (Python 3.9.7)

The code below was run on a HPC environment that did not support running Jupyter Lab via a browser.

As such, key console outputs were pasted as comments. Visualizations were outputted as pdfs or pngs.

A tsv file was exported at the end for overlaying HA-binding specificity on the UMAPs in the R script.

## Load packages & config

In [ ]:
from pathlib import Path
import os
import copy
import re
import math
import numpy as np
import pandas as pd
import scanpy as sc
from anndata import read_h5ad
from anndata import AnnData
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# scanpy settings

# verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.verbosity = 3             
sc.logging.print_header()

In [ ]:
# scanpy==1.8.2 anndata==0.9.1 umap==0.5.3 numpy==1.20.3 scipy==1.7.2 pandas==1.3.4 scikit-learn==1.0.1 statsmodels==0.13.1 python-igraph==0.10.6 pynndescent==0.5.10

In [ ]:
sc.settings.set_figure_params(dpi=120, dpi_save=150, vector_friendly=False, format="pdf", 
                              transparent=False, facecolor="w", color_map="viridis_r")

In [ ]:
sc.settings.figdir = "."

In [ ]:
# resolution for clustering all cells
res_1 = 0.12
anno_col_1 = f"anno_leiden_{res_1:.2f}"

In [ ]:
# resolution for clustering B cells
res_2 = 0.95
anno_col_2 = f"anno_leiden_{res_2:.2f}"

In [ ]:
# number of principal components to use
N_PC = 20

In [ ]:
# type of embedding
eb = "umap"

In [ ]:
# filenames
fn_1 = "WU397_matz_et_al_nat_imm_2026_gex_all_cells.h5ad"
fn_2 = "WU397_matz_et_al_nat_imm_2026_gex_b_cells.h5ad"
fn_save = f"WU397_matz_et_al_nat_imm_2026_gex_b_cell_{eb}.tsv.gz"

## All cells

### AnnData containing clustering results of all cells

In [ ]:
adata_1 = read_h5ad(fn_1)

In [ ]:
# reset .X (previously set to `None` in order to reduce file size)
adata_1.X = adata_1.layers["log_norm"]

In [ ]:
# check that column containing annotation labels is present
assert anno_col_1 in adata_1.obs.keys()

In [ ]:
adata_1.shape

In [ ]:
# (621494, 16539)
# 621877-383=621494 (383 removed for high PPBP expression)

### Cell counts by annotation

In [ ]:
adata_1.obs[anno_col_1].value_counts()

In [ ]:
# B           293316
# CD4+ T      250289
# CD8+ T       55624
# NK           10870
# Monocyte      9510
# pDC           1885
# Name: anno_leiden_0.12, dtype: int64

### Dot plot (Extended Data Fig 3b)

In [ ]:
genes_dict_1 = {
    "B": ["MS4A1", "CD19", "CD79A"],
    "T": ["CD3D", "CD3E", "CD3G", "IL7R"],
    "CD4+ T": ["CD4"],
    "CD8+ T": ["CD8A"],
    "NK": ["GZMB", "GNLY", "NKG7", "NCAM1"],
    "Monocyte": ["CD14", "LYZ", "CST3", "MS4A7"],
    "pDC": ["IL3RA", "CLEC4C"]
}

genes_lst_1 = [x for v in genes_dict_1.values() for x in v]

In [ ]:
# check that all genes for visualization are present in count matrix
assert all( [x in adata_1.var["gene_name"] for x in genes_lst_1] )

In [ ]:
anno_order_1 = ["B", "CD4+ T", "CD8+ T", "NK", "Monocyte", "pDC"]

In [ ]:
cur_fig = sc.pl.dotplot(adata_1, layer="log_norm", var_names=genes_dict_1, groupby=anno_col_1,
                        dendrogram=False, 
                        categories_order=anno_order_1, swap_axes=False,
                        cmap="Blues", return_fig=True, save=False)
cur_fig.savefig("ed_fig_3b.pdf", bbox_inches="tight")
plt.close()

### UMAPs (Extended Data Fig 3a)

In [ ]:
# color palette
anno_palette_1 = {
    "B": "violet", 
    "CD4+ T": "dodgerblue",
    "CD8+ T": "orange",
    "NK": "purple", 
    "Monocyte": "seagreen", 
    "pDC": "darkgray"
}

In [ ]:
# combined (Extended Data Fig 3a, left)
cur_fig = sc.pl.embedding(adata_1, basis=f"X_{eb}", color=anno_col_1, 
                          size=3, palette=anno_palette_1, 
                          legend_loc="right", legend_fontsize=0, legend_fontoutline=0,
                          frameon=True, ncols=1, title="",
                          return_fig=True, save=False)

cur_fig.savefig("ed_fig_3a_left.png", dpi=500, bbox_inches="tight")

plt.close(cur_fig)

In [ ]:
# DataFrame containing UMAP coordinates and select metadata columns
eb_df_anno_1 = pd.concat(
    [ pd.DataFrame(adata_1.obsm[f"X_{eb}"], columns=[f"{eb}_x", f"{eb}_y"], index=adata_1.obs.index),
      adata_1.obs.loc[:, [anno_col_1, "donor"]] ], 
    axis=1)

In [ ]:
# remove grid lines and ticks
sns.set_style("white")

In [ ]:
# stratified by donor (Extended Data Fig 3a, right)

max_per_row = 4

# a Facet.Grid
cur_fig = sns.relplot(x=f"{eb}_x", y=f"{eb}_y", data=eb_df_anno_1, 
                      hue=anno_col_1, palette=anno_palette_1,
                      hue_order=anno_order_1, # legend order
                      col="donor", col_wrap=max_per_row,
                      s=3, alpha=0.85, 
                      height=5, aspect=0.9)

# remove x-axis label
cur_fig.set(xlabel=None)
# remove y-axis label
cur_fig.set(ylabel=None)
# remove tick labels
cur_fig.set(xticklabels=[])
cur_fig.set(yticklabels=[])

# set legend title
cur_fig._legend.set_title("")


cur_fig.savefig("ed_fig_3a_right.png", dpi=120, bbox_inches="tight")

plt.close()

In [ ]:
del adata_1, eb_df_anno_1

## B cells

### AnnData containing clustering results of B cells

In [ ]:
adata_2 = read_h5ad(fn_2)

In [ ]:
# reset .X (previously set to `None` in order to reduce file size)
adata_2.X = adata_2.layers["log_norm"]

In [ ]:
# check that column containing annotation labels is present
assert anno_col_2 in adata_2.obs.keys()

In [ ]:
adata_2.shape

In [ ]:
# (280493, 16539)

### Cell counts by annotation

In [ ]:
adata_2.obs[anno_col_2].value_counts()

In [ ]:
# MBC      144561
# PB        61593
# Naive     59622
# GC         6808
# ABC        5514
# LNPC       2395
# Name: anno_leiden_0.95, dtype: int64

### Dot plot (Extended Data Fig 3d)

In [ ]:
genes_dict_2 = {
    "GC": ["BCL6", "RGS13", "MEF2B", "STMN1", "ELL3", "SERPINA9"],
    "PB/LNPC": ["XBP1", "IRF4", "SEC11C", "FKBP11", "JCHAIN", "PRDM1"],
    "ABC": ["TBX21", "FCRL5", "ITGAX", "NKG7", "ZEB2", "CR2"],
    "Naive": ["TCL1A", "IL4R", "CCR7", "IGHM", "IGHD"],
    "MBC": ["TNFRSF13B", "CD27", "CD24"]
}

genes_lst_2 = [x for v in genes_dict_2.values() for x in v]

In [ ]:
# check that all genes for visualization are present in count matrix
assert all( [x in adata_2.var["gene_name"] for x in genes_lst_2] )

In [ ]:
anno_order_2 = ["Naive", "PB", "GC", "LNPC", "ABC", "MBC"]

In [ ]:
cur_fig = sc.pl.dotplot(adata_2, layer="log_norm", var_names=genes_dict_2, groupby=anno_col_2,
                        dendrogram=False, 
                        categories_order=anno_order_2, swap_axes=False,
                        cmap="Blues", return_fig=True, save=False)
cur_fig.savefig("ed_fig_3d.pdf", bbox_inches="tight")
plt.close()

### UMAPs (Extended Data Fig 3c)

In [ ]:
# color palette
anno_palette_2 = {
    "GC": "dodgerblue", 
    "LNPC": "forestgreen",
    "PB": "red",
    "Naive": "orange",
    "ABC": "mediumspringgreen",
    "MBC": "violet"
}

In [ ]:
# combined (Extended Data Fig 3c, left)
cur_fig = sc.pl.embedding(adata_2, basis=f"X_{eb}", color=anno_col_2, 
                          size=3, palette=anno_palette_2, 
                          legend_loc="right", legend_fontsize=0, legend_fontoutline=0,
                          frameon=True, ncols=1, title="",
                          return_fig=True, save=False)

cur_fig.savefig("ed_fig_3c_left.png", dpi=500, bbox_inches="tight")

plt.close(cur_fig)

In [ ]:
# DataFrame containing UMAP coordinates and select metadata columns
eb_df_anno_2 = pd.concat(
    [ pd.DataFrame(adata_2.obsm[f"X_{eb}"], columns=[f"{eb}_x", f"{eb}_y"], index=adata_2.obs.index),
      adata_2.obs.loc[:, ["cell_id", "donor", "timepoint", "tissue", "sorting", anno_col_2]] ], 
    axis=1)

In [ ]:
# remove grid lines and ticks
sns.set_style("white")

In [ ]:
# stratified by donor (Extended Data Fig 3c, right)

max_per_row = 4

# a Facet.Grid
cur_fig = sns.relplot(x=f"{eb}_x", y=f"{eb}_y", data=eb_df_anno_2, 
                      hue=anno_col_2, palette=anno_palette_2,
                      hue_order=anno_order_2, # legend order
                      col="donor", col_wrap=max_per_row,
                      s=3, alpha=0.85, 
                      height=5, aspect=0.9)

# remove x-axis label
cur_fig.set(xlabel=None)
# remove y-axis label
cur_fig.set(ylabel=None)
# remove tick labels
cur_fig.set(xticklabels=[])
cur_fig.set(yticklabels=[])

# set legend title
cur_fig._legend.set_title("")


cur_fig.savefig("ed_fig_3c_right.png", dpi=120, bbox_inches="tight")

plt.close()

### Export DataFrame

In [ ]:
# used in R script for UMAP-based visualizations
eb_df_anno_2.to_csv(fn_save, sep="\t", header=True, index=False, compression="gzip")